# Figure 6 | Integrated Aβ-associated NVU remodeling

Cleaned notebook for the public NVU AD spatial transcriptomics repository. Paths are repository-relative; set `NVU_PROJECT_ROOT` if running from another working directory.


In [ ]:
# Repository-relative paths
PROJECT_ROOT <- Sys.getenv("NVU_PROJECT_ROOT", unset = normalizePath(file.path(getwd(), ".."), mustWork = FALSE))
DATA_DIR <- file.path(PROJECT_ROOT, "data")
RESULTS_DIR <- file.path(PROJECT_ROOT, "results")
REFERENCE_DIR <- file.path(PROJECT_ROOT, "references")
FIGURE_DIR <- file.path(RESULTS_DIR, "figure6")
dir.create(FIGURE_DIR, recursive = TRUE, showWarnings = FALSE)


In [ ]:
NVU = readRDS('../results/05_nvu_analysis_results/Hip/200/Hip_merged-subtype_0330.rds')

In [ ]:
library(dplyr)
library(tidyr)
library(FNN)
library(ggplot2)
library(purrr)

### Aβ细胞类型 ✅️

In [ ]:
# ============================================================
# Abeta 周围 NVU 密度分析 —— 最终版
#
# 设计：
#   AD-Abeta   : 真实斑块，形态学距离，100µm内
#   AD-Control : AD样本随机点，圆形面积，100µm内
#   Con-Control: Con样本随机点，圆形面积，100µm内
#
# NVU_unit 密度计算逻辑（per-sample汇总）：
#   分子：某sample下所有斑块/对照点100µm范围内的 unique unit_id 总数
#   分母：该sample下所有斑块/对照点的圆形面积之和
#
# 输出：
#   A. 单区间bar图（0-100µm整体密度，三组并排）
#   B. 折线图（每20µm一bin）
# ============================================================

group_colors <- c(
  "AD-Abeta"    = "#E64B35",
  "AD-Control"  = "#F39B7F",
  "Con-Control" = "#4DBBD5"
)



setwd('../results/figure6')

# ════════════════════════════════════════════════════════════
# 0. 全局参数
# ════════════════════════════════════════════════════════════

AD_samples <- c("D03556C4", "D01574C4", "C01834C3", "D04303A6",
                "D03556E4", "D01574B6", "D03556D4", "D04303D1",
                "D03556E6", "C01840B1")#,'AD_1','AD_2'


Con_samples <- unique(c("D01574A6", "D01574B1", #"D01574C2", "D01574C6",
                         "B03421A5", "B03421A6", "D03556D6", "D03556E2",
                         "D04305A6", "D04305A4", "C04595E2", "C04595F1",
                         "D03556F6", "D03556F4",'control_1','control_2'))#

all_samples      <- c(AD_samples, Con_samples)
target_celltypes <- c("Micro_2", "Astro_2", "Astro_3")
nvu_celltypes    <- c("Neuron", "Astro", "Micro", "Endo", "Pericyte")
ct_list          <- c("Micro_2", "Astro_2", "Astro_3", "NVU_unit")
target_regions   <- c("CA1", "SLRM", "FAS")

coord_to_um  <- 0.5          # 1坐标单位 = 0.5µm
coord_to_um2 <- coord_to_um^2

# 距离参数
max_um    <- 100
max_coord <- max_um / coord_to_um   # = 200 坐标单位


# 对照组参数
control_min_dist <- 600   # 距真实斑块最小坐标距离（AD-Control用）
n_control        <- 25    # 每sample随机中心数
n_candidates     <- 5000

# 面积格点步长（AD-Abeta真实斑块用）
area_grid_step   <- 2
boundary_grid_px <- 3

# ════════════════════════════════════════════════════════════
# 0b. 衍生参数（依赖上面的全局参数）
# ════════════════════════════════════════════════════════════

# ── bin 定义（每20µm一档，共5档）─────────────────────────
bin_width_um    <- 20
bin_width_coord <- bin_width_um / coord_to_um          # = 40 坐标单位
n_bins          <- max_um / bin_width_um                # = 5

bin_breaks_coord <- seq(0, max_coord, by = bin_width_coord)   # 0,40,80,120,160,200
bin_labels       <- paste0(seq(0, max_um - bin_width_um, by = bin_width_um),
                            "-",
                            seq(bin_width_um, max_um, by = bin_width_um),
                            "µm")
# "0-20µm"  "20-40µm" "40-60µm" "60-80µm" "80-100µm"

# ── 单圆 / 圆环面积（µm²）──────────────────────────────
single_area_um2 <- pi * max_um^2                        # 100µm 整圆

bin_ring_areas <- tibble(
  bin_label    = bin_labels,
  r_inner_um   = seq(0,           max_um - bin_width_um, by = bin_width_um),
  r_outer_um   = seq(bin_width_um, max_um,               by = bin_width_um),
  bin_area_um2 = pi * (r_outer_um^2 - r_inner_um^2)
)
set.seed(421)

abeta_filled_path <- function(s)
  paste0("../data/abeta/", s, "_abeta_filled.csv")

# ════════════════════════════════════════════════════════════
# 1. 加载细胞数据
# ════════════════════════════════════════════════════════════

load_cells <- function(s) {
  message("Loading: ", s)
  subset(NVU, sample_id == s)@meta.data %>%
    select(cellid, x, y, subcelltype, area_m, unit_id) %>%
    mutate(sample       = s,
           sample_group = ifelse(s %in% AD_samples, "AD", "Con"))
}

all_cells_raw <- lapply(all_samples, load_cells) %>% bind_rows()

# ════════════════════════════════════════════════════════════
# 2. 加载真实斑块像素（仅AD）
# ════════════════════════════════════════════════════════════

message("\n=== 加载斑块像素 ===")

all_abeta_filled <- lapply(AD_samples, function(s) {
  f <- abeta_filled_path(s)
  if (!file.exists(f)) { warning("缺文件: ", f); return(NULL) }
  df <- read.csv(f)
  if (!"abeta_id" %in% names(df) && "plaque_id" %in% names(df))
    df <- rename(df, abeta_id = plaque_id)
  df %>%
    mutate(sample    = s,
           plaque_id = paste0(s, "_p", abeta_id)) %>%
    select(sample, plaque_id, abeta_id, sp_x, sp_y, n_pixels)
}) %>% bind_rows()

message("斑块总数: ", n_distinct(all_abeta_filled$plaque_id))

# 边界稀疏化
thin_boundary <- function(df, gpx = boundary_grid_px) {
  df %>%
    mutate(gx = round(sp_x / gpx), gy = round(sp_y / gpx)) %>%
    distinct(gx, gy, .keep_all = TRUE) %>%
    select(-gx, -gy)
}

abeta_boundary_list <- all_abeta_filled %>%
  group_by(plaque_id) %>% group_split() %>%
  lapply(thin_boundary) %>%
  setNames(sapply(., function(d) d$plaque_id[1]))

all_centroids <- all_abeta_filled %>%
  group_by(sample, plaque_id) %>%
  summarise(cx = mean(sp_x), cy = mean(sp_y), .groups = "drop")

# ════════════════════════════════════════════════════════════
# 3. assign_bin：距离 → bin标签
# ════════════════════════════════════════════════════════════

assign_bin <- function(dist_coord) {
  idx <- findInterval(dist_coord, bin_breaks_coord,
                      rightmost.closed = FALSE, left.open = TRUE)
  ifelse(dist_coord <= 0 | idx < 1 | idx > n_bins,
         NA_character_, bin_labels[idx])
}

# ════════════════════════════════════════════════════════════
# 4. AD-Abeta：形态学距离分配
# ════════════════════════════════════════════════════════════

assign_dist_AD <- function(s) {
  message("  AD形态学距离: ", s)
  cells_s   <- all_cells_raw %>% filter(sample == s, area_m %in% target_regions)
  centers_s <- all_centroids  %>% filter(sample == s)
  if (nrow(centers_s) == 0 || nrow(cells_s) == 0)
    return(cells_s %>% mutate(nearest_plaque   = NA_character_,
                               min_dist_to_edge = Inf,
                               bin_label        = NA_character_))

  nn_c <- FNN::get.knnx(as.matrix(centers_s[, c("cx","cy")]),
                          as.matrix(cells_s[,  c("x", "y")]), k = 1)
  cells_s$nearest_plaque <- centers_s$plaque_id[nn_c$nn.index[, 1]]

  split(cells_s, cells_s$nearest_plaque) %>%
    imap_dfr(function(cg, pid) {
      bpts <- abeta_boundary_list[[pid]]
      if (is.null(bpts) || nrow(bpts) == 0) {
        cg$min_dist_to_edge <- Inf; cg$bin_label <- NA_character_; return(cg)
      }
      nn_f <- FNN::get.knnx(as.matrix(bpts[, c("sp_x","sp_y")]),
                              as.matrix(cg[,  c("x",   "y")]),   k = 1)
      cg$min_dist_to_edge <- nn_f$nn.dist[, 1]
      cg$bin_label        <- assign_bin(cg$min_dist_to_edge)
      cg
    })
}

message("\n=== AD 距离分配 ===")
AD_cells <- lapply(AD_samples, assign_dist_AD) %>% bind_rows()

Con_cells <- all_cells_raw %>%
  filter(sample %in% Con_samples, area_m %in% target_regions) %>%
  mutate(nearest_plaque   = NA_character_,
         min_dist_to_edge = Inf,
         bin_label        = NA_character_)

# ════════════════════════════════════════════════════════════
# 5. AD-Abeta 真实斑块面积（格点法，仅非NVU细胞类型使用）
# ════════════════════════════════════════════════════════════

compute_plaque_areas <- function(step = area_grid_step) {
  message("\n=== 真实斑块面积计算 ===")
  pids       <- names(abeta_boundary_list)
  res_single <- vector("list", length(pids))
  res_bin    <- vector("list", length(pids))
  pb <- txtProgressBar(0, length(pids), style = 3)

  for (i in seq_along(pids)) {
    pid    <- pids[i]
    filled <- all_abeta_filled %>% filter(plaque_id == pid)
    if (nrow(filled) == 0) { setTxtProgressBar(pb, i); next }

    pts   <- as.matrix(filled[, c("sp_x","sp_y")])
    x_seq <- seq(min(filled$sp_x) - max_coord, max(filled$sp_x) + max_coord, by = step)
    y_seq <- seq(min(filled$sp_y) - max_coord, max(filled$sp_y) + max_coord, by = step)
    gpts  <- as.matrix(expand.grid(x_seq, y_seq))
    dist  <- FNN::get.knnx(pts, gpts, k = 1)$nn.dist[, 1]
    ua    <- step^2 * coord_to_um2

    res_single[[i]] <- tibble(
      plaque_id     = pid,
      ring_area_um2 = sum(dist > 0 & dist <= max_coord) * ua
    )
    bv    <- assign_bin(dist)
    valid <- !is.na(bv)
    if (any(valid)) {
      cnt <- table(bv[valid])
      res_bin[[i]] <- tibble(plaque_id    = pid,
                              bin_label    = names(cnt),
                              bin_area_um2 = as.numeric(cnt) * ua)
    }
    setTxtProgressBar(pb, i)
  }
  close(pb)
  list(single = bind_rows(res_single), bin = bind_rows(res_bin))
}

plaque_areas       <- compute_plaque_areas()
plaque_single_area <- plaque_areas$single
plaque_bin_area    <- plaque_areas$bin

# ════════════════════════════════════════════════════════════
# 6. AD-Abeta 密度计算
#
#  非NVU细胞：每斑块单独算密度（per-plaque），分母为单斑块圆形面积
#  NVU_unit ：per-sample 汇总，分子为sample内所有斑块范围内unique unit总数
#              分母为该sample所有斑块圆形面积之和
# ════════════════════════════════════════════════════════════

calc_abeta_density <- function(cells_df, ct_label, use_unit = FALSE) {

  if (use_unit) {
    # ── NVU_unit single：per-sample × area_m ─────────────────
    # 分母：该 sample × area_m 下的斑块数 × 单斑块圆形面积
    n_plaques_per_sample <- cells_df %>%
      filter(!is.na(nearest_plaque)) %>%
      group_by(sample, area_m) %>%
      summarise(n_plaques = n_distinct(nearest_plaque), .groups = "drop") %>%
      mutate(total_area_um2 = n_plaques * single_area_um2)

    d_s <- cells_df %>%
      filter(min_dist_to_edge > 0, min_dist_to_edge <= max_coord,
             !is.na(nearest_plaque),
             subcelltype %in% nvu_celltypes,
             !is.na(unit_id), unit_id != "") %>%
      group_by(sample, area_m) %>%
      summarise(n_units = n_distinct(unit_id), .groups = "drop") %>%
      filter(n_units > 0) %>%
      left_join(n_plaques_per_sample, by = c("sample", "area_m")) %>%
      filter(!is.na(total_area_um2), total_area_um2 > 0) %>%
      mutate(density     = n_units / total_area_um2 * 1e6,
             subcelltype = ct_label,
             group       = "AD-Abeta",
             mode        = "single",
             bin_label   = "0-100µm",
             center_id   = "per_sample") %>%
      select(sample, area_m, center_id, subcelltype, density, group, mode, bin_label)

    # ── NVU_unit bin：per-sample × area_m × bin_label ────────
    # 每个unit取距其最近斑块边缘最近的那条记录，确定所属bin
    cells_unit_binned <- cells_df %>%
      filter(!is.na(nearest_plaque),
             subcelltype %in% nvu_celltypes,
             !is.na(unit_id), unit_id != "",
             !is.na(bin_label)) %>%
      group_by(sample, area_m, unit_id) %>%
      slice_min(min_dist_to_edge, n = 1, with_ties = FALSE) %>%
      ungroup()

    # 分母：该 sample × area_m × bin 下斑块数 × 圆环面积
    n_plaques_per_bin <- cells_df %>%
      filter(!is.na(nearest_plaque), !is.na(bin_label)) %>%
      group_by(sample, area_m, bin_label) %>%
      summarise(n_plaques = n_distinct(nearest_plaque), .groups = "drop") %>%
      left_join(bin_ring_areas %>% select(bin_label, bin_area_um2),
                by = "bin_label") %>%
      mutate(total_area_um2 = n_plaques * bin_area_um2)

    d_b <- cells_unit_binned %>%
      group_by(sample, area_m, bin_label) %>%
      summarise(n_units = n_distinct(unit_id), .groups = "drop") %>%
      filter(n_units > 0) %>%
      left_join(n_plaques_per_bin, by = c("sample", "area_m", "bin_label")) %>%
      filter(!is.na(total_area_um2), total_area_um2 > 0) %>%
      mutate(density     = n_units / total_area_um2 * 1e6,
             subcelltype = ct_label,
             group       = "AD-Abeta",
             mode        = "bin",
             center_id   = "per_sample") %>%
      select(sample, area_m, center_id, subcelltype, density, group, mode, bin_label)

    return(bind_rows(d_s, d_b))
  }

  # ── 非 NVU_unit：per-plaque 逻辑不变 ─────────────────────
  count_var <- "cellid"

  d_s <- cells_df %>%
    filter(min_dist_to_edge > 0, min_dist_to_edge <= max_coord,
           !is.na(nearest_plaque)) %>%
    group_by(sample, area_m, nearest_plaque) %>%
    summarise(n_cells = n_distinct(.data[[count_var]]), .groups = "drop") %>%
    filter(n_cells > 0) %>%
    mutate(density     = n_cells / single_area_um2 * 1e6,
           subcelltype = ct_label,
           group       = "AD-Abeta",
           mode        = "single",
           bin_label   = "0-100µm",
           center_id   = nearest_plaque) %>%
    select(sample, area_m, center_id, subcelltype, density, group, mode, bin_label)

  d_b <- cells_df %>%
    filter(!is.na(bin_label), !is.na(nearest_plaque)) %>%
    group_by(sample, area_m, nearest_plaque, bin_label) %>%
    summarise(n_cells = n_distinct(.data[[count_var]]), .groups = "drop") %>%
    filter(n_cells > 0) %>%
    left_join(bin_ring_areas %>% select(bin_label, bin_area_um2),
              by = "bin_label") %>%
    filter(!is.na(bin_area_um2), bin_area_um2 > 0) %>%
    mutate(density     = n_cells / bin_area_um2 * 1e6,
           subcelltype = ct_label,
           group       = "AD-Abeta",
           mode        = "bin",
           center_id   = nearest_plaque) %>%
    select(sample, area_m, center_id, subcelltype, density, group, mode, bin_label)

  bind_rows(d_s, d_b)
}

# 构建 abeta_all
# NVU_unit 传入完整 AD_cells，函数内部按 nvu_celltypes 过滤
abeta_all <- bind_rows(
  AD_cells %>% filter(subcelltype == "Micro_2") %>% calc_abeta_density("Micro_2"),
  AD_cells %>% filter(subcelltype == "Astro_2") %>% calc_abeta_density("Astro_2"),
  AD_cells %>% filter(subcelltype == "Astro_3") %>% calc_abeta_density("Astro_3"),
  AD_cells %>% calc_abeta_density("NVU_unit", use_unit = TRUE)
)

message("abeta_all 行数: ", nrow(abeta_all))
print(table(abeta_all$subcelltype, abeta_all$mode))



# ════════════════════════════════════════════════════════════
# 7. 对照组随机中心点（按 area_m 分区生成）
# ════════════════════════════════════════════════════════════

get_ctrl_centers <- function(s, is_AD) {
  grp_lab <- ifelse(is_AD, "AD-Control", "Con-Control")
  result  <- vector("list", length(target_regions))

  for (k in seq_along(target_regions)) {
    am      <- target_regions[k]
    cells_s <- all_cells_raw %>%
      filter(sample == s, area_m == am)           # ← 按 area_m 过滤
    if (nrow(cells_s) == 0) next

    margin  <- max_coord + bin_width_coord * 2
    x_valid <- range(cells_s$x) + c(margin, -margin)
    y_valid <- range(cells_s$y) + c(margin, -margin)
    if (x_valid[1] >= x_valid[2] || y_valid[1] >= y_valid[2]) next

    cands <- tibble(
      ctrl_x = runif(n_candidates, x_valid[1], x_valid[2]),
      ctrl_y = runif(n_candidates, y_valid[1], y_valid[2]),
      area_m = am                                  # ← 记录 area_m
    )

    if (is_AD) {
      cs <- all_centroids %>% filter(sample == s)
      if (nrow(cs) > 0) {
        nn    <- FNN::get.knnx(as.matrix(cs[, c("cx","cy")]),
                                as.matrix(cands[, c("ctrl_x","ctrl_y")]), k = 1)
        cands <- cands %>% filter(nn$nn.dist[, 1] > control_min_dist)
      }
    }
    if (nrow(cands) == 0) next

    n_take <- min(n_control, nrow(cands))
    chosen <- cands %>% slice_sample(n = n_take)
    message("  ", s, " (", grp_lab, ") ", am, ": ", n_take, " 个中心点")

    result[[k]] <- tibble(
      sample    = s,
      area_m    = am,                              # ← 对照点携带 area_m
      center_id = paste0(s, "_", am, "_ctrl_", seq_len(n_take)),
      ctrl_x    = chosen$ctrl_x,
      ctrl_y    = chosen$ctrl_y,
      group     = grp_lab
    )
  }
  bind_rows(result)
}

AD_ctrl_centers  <- lapply(AD_samples,  function(s) get_ctrl_centers(s, TRUE))  %>% bind_rows()
Con_ctrl_centers <- lapply(Con_samples, function(s) get_ctrl_centers(s, FALSE)) %>% bind_rows()
all_ctrl_centers <- bind_rows(AD_ctrl_centers, Con_ctrl_centers)
                           
# ════════════════════════════════════════════════════════════
# 8. 对照组密度计算
#
#  非NVU细胞：每对照点单独算密度（per-center），分母为单个圆形面积
#  NVU_unit ：per-sample 汇总，函数循环结束后单独计算
#              分子：该sample所有对照点100µm范围内的 unique unit_id 总数
#              分母：该sample对照点数 × 单圆面积
# ════════════════════════════════════════════════════════════

message("\n=== 对照组密度计算 ===")

compute_ctrl_density <- function(ctrl_centers_df, cells_pool) {
  grp <- unique(ctrl_centers_df$group)
  message("  处理: ", grp)

  # ── Part A：非NVU细胞 per-center 循环 ─────────────────────
  result <- vector("list", nrow(ctrl_centers_df))

  for (i in seq_len(nrow(ctrl_centers_df))) {
    row   <- ctrl_centers_df[i, ]
    s     <- row$sample
    cid   <- row$center_id
    cx    <- row$ctrl_x
    cy    <- row$ctrl_y
    grp_i <- row$group

    cells_s <- cells_pool %>% filter(sample == s, area_m == row$area_m)
    if (nrow(cells_s) == 0) next

    cells_s$dist_to_center <- sqrt((cells_s$x - cx)^2 + (cells_s$y - cy)^2)
    cells_s$bin_label      <- assign_bin(cells_s$dist_to_center)

    # 单区间：非NVU细胞
    cells_100 <- cells_s %>%
      filter(dist_to_center > 0, dist_to_center <= max_coord)

    single_non_nvu <- cells_100 %>%
      group_by(area_m) %>%
      group_split() %>%
      lapply(function(df_area) {
        am <- unique(df_area$area_m)
        bind_rows(
          { n <- sum(df_area$subcelltype == "Micro_2")
            if (n > 0) tibble(sample=s, area_m=am, center_id=cid,
                               subcelltype="Micro_2",
                               density=n/single_area_um2*1e6,
                               group=grp_i, mode="single", bin_label="0-100µm") },
          { n <- sum(df_area$subcelltype == "Astro_2")
            if (n > 0) tibble(sample=s, area_m=am, center_id=cid,
                               subcelltype="Astro_2",
                               density=n/single_area_um2*1e6,
                               group=grp_i, mode="single", bin_label="0-100µm") },
          { n <- sum(df_area$subcelltype == "Astro_3")
            if (n > 0) tibble(sample=s, area_m=am, center_id=cid,
                               subcelltype="Astro_3",
                               density=n/single_area_um2*1e6,
                               group=grp_i, mode="single", bin_label="0-100µm") }
        )
      }) %>% bind_rows()

    # 折线bin：非NVU细胞
    calc_bin_ctrl_cell <- function(df, ct) {
      df %>%
        filter(!is.na(bin_label)) %>%
        group_by(area_m, bin_label) %>%
        summarise(n_cells = n(), .groups = "drop") %>%
        left_join(bin_ring_areas %>% select(bin_label, bin_area_um2),
                  by = "bin_label") %>%
        filter(!is.na(bin_area_um2), bin_area_um2 > 0) %>%
        mutate(sample      = s,
               center_id   = cid,
               subcelltype = ct,
               density     = n_cells / bin_area_um2 * 1e6,
               group       = grp_i,
               mode        = "bin") %>%
        select(sample, area_m, center_id, subcelltype,
               density, group, mode, bin_label)
    }

    result[[i]] <- bind_rows(
      cells_s %>% filter(subcelltype == "Micro_2") %>% calc_bin_ctrl_cell("Micro_2"),
      cells_s %>% filter(subcelltype == "Astro_2") %>% calc_bin_ctrl_cell("Astro_2"),
      cells_s %>% filter(subcelltype == "Astro_3") %>% calc_bin_ctrl_cell("Astro_3"),
      single_non_nvu
    )
  }

  result_df <- bind_rows(result)

  # ── Part B：NVU_unit per-sample 汇总 ──────────────────────
# ── Part B：NVU_unit per-sample 汇总（按 area_m 分区匹配）──
nvu_result <- lapply(unique(ctrl_centers_df$sample), function(s) {
  grp_i <- unique(ctrl_centers_df$group[ctrl_centers_df$sample == s])

  lapply(target_regions, function(am) {
    # 对照点只取当前 area_m 的
    centers_am <- ctrl_centers_df %>% filter(sample == s, area_m == am)
    cells_am   <- cells_pool %>%
      filter(sample == s, area_m == am,
             subcelltype %in% nvu_celltypes,
             !is.na(unit_id), unit_id != "")

    if (nrow(centers_am) == 0 || nrow(cells_am) == 0) return(NULL)

    nn <- FNN::get.knnx(
      as.matrix(centers_am[, c("ctrl_x","ctrl_y")]),
      as.matrix(cells_am[,  c("x",     "y")]), k = 1
    )
    cells_am$dist_to_nearest_ctrl <- nn$nn.dist[, 1]
    cells_am$bin_label_ctrl       <- assign_bin(cells_am$dist_to_nearest_ctrl)

    n_centers      <- nrow(centers_am)
    total_area_um2 <- n_centers * single_area_um2

    # single
    d_s <- cells_am %>%
      filter(dist_to_nearest_ctrl > 0,
             dist_to_nearest_ctrl <= max_coord) %>%
      summarise(n_units = n_distinct(unit_id)) %>%
      filter(n_units > 0) %>%
      mutate(sample=s, area_m=am, center_id="per_sample",
             subcelltype="NVU_unit",
             density=n_units/total_area_um2*1e6,
             group=grp_i, mode="single", bin_label="0-100µm") %>%
      select(sample, area_m, center_id, subcelltype,
             density, group, mode, bin_label)

    # bin
    d_b <- cells_am %>%
      filter(!is.na(bin_label_ctrl)) %>%
      group_by(unit_id) %>%
      slice_min(dist_to_nearest_ctrl, n=1, with_ties=FALSE) %>%
      ungroup() %>%
      group_by(bin_label_ctrl) %>%
      summarise(n_units = n_distinct(unit_id), .groups="drop") %>%
      rename(bin_label = bin_label_ctrl) %>%
      filter(n_units > 0) %>%
      left_join(bin_ring_areas %>% select(bin_label, bin_area_um2),
                by = "bin_label") %>%
      filter(!is.na(bin_area_um2), bin_area_um2 > 0) %>%
      mutate(total_area_um2_bin = n_centers * bin_area_um2,
             sample=s, area_m=am, center_id="per_sample",
             subcelltype="NVU_unit",
             density=n_units/total_area_um2_bin*1e6,
             group=grp_i, mode="bin") %>%
      select(sample, area_m, center_id, subcelltype,
             density, group, mode, bin_label)

    bind_rows(d_s, d_b)
  }) %>% bind_rows()
}) %>% bind_rows()

  bind_rows(result_df, nvu_result)
}

ctrl_all <- bind_rows(
  compute_ctrl_density(AD_ctrl_centers,  AD_cells),
  compute_ctrl_density(Con_ctrl_centers, Con_cells)
)

message("ctrl_all 行数: ", nrow(ctrl_all))
print(table(ctrl_all$group, ctrl_all$mode, ctrl_all$subcelltype))

# ════════════════════════════════════════════════════════════
# 9. 合并 & 两步聚合
# ════════════════════════════════════════════════════════════

all_combined <- bind_rows(abeta_all, ctrl_all) %>%
  mutate(
    group       = factor(group,
                         levels = c("AD-Abeta","AD-Control","Con-Control")),
    subcelltype = factor(subcelltype,
                         levels = c("Micro_2","Astro_2","Astro_3","NVU_unit")),
    area_m      = factor(area_m, levels = target_regions),
    bin_label   = factor(bin_label, levels = c("0-100µm", bin_labels))
  )

# 第一步：per-sample 均值
# 注意：NVU_unit 的 AD-Abeta / 对照组 per-sample 记录已经是 sample level，
# 此步对它们不产生额外平均，仅对非NVU细胞（per-plaque / per-center）做样本内均值
per_sample <- all_combined %>%
  group_by(group, area_m, subcelltype, mode, bin_label, sample) %>%
  summarise(sample_mean = mean(density, na.rm = TRUE), .groups = "drop")

# 第二步：跨样本均值 ± SEM
summary_all <- per_sample %>%
  group_by(group, area_m, subcelltype, mode, bin_label) %>%
  summarise(
    mean_density = mean(sample_mean, na.rm = TRUE),
    sem          = sd(sample_mean,   na.rm = TRUE) / sqrt(n()),
    n_samples    = n(),
    .groups      = "drop"
  ) %>%
  mutate(
    group       = factor(group,       levels = c("AD-Abeta","AD-Control","Con-Control")),
    subcelltype = factor(subcelltype, levels = c("Micro_2","Astro_2","Astro_3","NVU_unit")),
    area_m      = factor(area_m,      levels = target_regions),
    bin_label   = factor(bin_label,   levels = c("0-100µm", bin_labels))
  )

message("\n=== 诊断：单区间各组密度均值 ===")
summary_all %>%
  filter(mode == "single") %>%
  group_by(group, subcelltype) %>%
  summarise(mean_across_regions = mean(mean_density, na.rm = TRUE),
            .groups = "drop") %>%
  print()

# ════════════════════════════════════════════════════════════
# 10. 绘图：单区间 bar 图（含显著性标注）
# ════════════════════════════════════════════════════════════

plot_single_manual <- function(ct) {
  dat <- summary_all %>%
    filter(mode == "single", subcelltype == ct, !is.na(mean_density))

  raw_dat <- per_sample %>%
    filter(mode == "single", subcelltype == ct, !is.na(sample_mean))

  group_lvls  <- c("AD-Abeta","AD-Control","Con-Control")
  area_lvls   <- c("CA1","SLRM","FAS")
  dodge_width <- 0.8
  offsets     <- c(-dodge_width/3, 0, dodge_width/3)
  names(offsets) <- group_lvls

  # 至少3个样本才做检验
  valid_groups <- raw_dat %>%
    group_by(area_m, group) %>%
    summarise(n = sum(!is.na(sample_mean)), .groups = "drop") %>%
    filter(n >= 3)

  valid_areas <- valid_groups %>%
    group_by(area_m) %>%
    filter(n_distinct(group) == 3) %>%
    pull(area_m) %>%
    unique()

  stat_res <- tibble()

  if (length(valid_areas) > 0) {
    stat_res <- tryCatch({
      raw_dat %>%
        filter(area_m %in% valid_areas) %>%
        mutate(area_m = as.character(area_m),
               group  = as.character(group)) %>%
        group_by(area_m) %>%
        group_modify(function(df, key) {
          pairs <- list(
            c("AD-Abeta",   "AD-Control"),
            c("AD-Abeta",   "Con-Control"),
            c("AD-Control", "Con-Control")
          )
          purrr::map_dfr(pairs, function(pr) {
            g1 <- df$sample_mean[df$group == pr[1]]
            g2 <- df$sample_mean[df$group == pr[2]]
            if (length(g1) < 2 || length(g2) < 2 ||
                all(is.na(g1)) || all(is.na(g2))) return(NULL)
            if (length(unique(c(g1, g2))) < 2) return(NULL)
            tryCatch({
              p_val <- wilcox.test(g1, g2, exact = FALSE)$p.value
              tibble(group1 = pr[1], group2 = pr[2], p = p_val)
            }, error = function(e) NULL)
          })
        }) %>%
        ungroup() %>%
        group_by(area_m) %>%
        mutate(p.adj = p.adjust(p, method = "BH")) %>%
        ungroup() %>%
        mutate(
          p.adj.signif = case_when(
            p.adj < 0.001 ~ "***",
            p.adj < 0.01  ~ "**",
            p.adj < 0.05  ~ "*",
            TRUE          ~ "ns"
          )
        ) %>%
        filter(p.adj.signif != "ns")
    }, error = function(e) {
      message("检验失败 (", ct, "): ", conditionMessage(e))
      tibble()
    })
  }

  if (nrow(stat_res) > 0) {
    stat_res <- stat_res %>%
      mutate(
        area_x  = as.numeric(factor(area_m, levels = area_lvls)),
        x_start = area_x + offsets[group1],
        x_end   = area_x + offsets[group2]
      ) %>%
      group_by(area_m) %>%
      mutate(
        y_max      = max(dat$mean_density + dat$sem, na.rm = TRUE),
        y.position = y_max * (1.08 + 0.10 * (row_number() - 1))
      ) %>%
      ungroup()
  }

  p <- ggplot(dat, aes(x = area_m, y = mean_density, fill = group)) +
    geom_col(position = position_dodge(dodge_width), width = 0.75) +
    geom_errorbar(
      aes(ymin = pmax(mean_density - sem, 0), ymax = mean_density + sem),
      position = position_dodge(dodge_width), width = 0.3, linewidth = 0.5
    ) +
    scale_fill_manual(values = group_colors) +
    scale_x_discrete(limits = area_lvls) +
    scale_y_continuous(expand = expansion(mult = c(0, 0.28))) +
    labs(
      title = ct,
      x     = "Brain region",
      y     = expression("Density (cells·mm"^{-2}*")"),
      fill  = NULL
    ) +
    theme_classic(base_size = 13) +
    theme(
      legend.position = "bottom",
      axis.line       = element_line(linewidth = 0.5),
      plot.title      = element_text(face = "bold", hjust = 0.5)
    )

  if (nrow(stat_res) > 0) {
    for (i in seq_len(nrow(stat_res))) {
      row <- stat_res[i, ]
      p <- p +
        annotate("segment",
                 x = row$x_start, xend = row$x_end,
                 y = row$y.position, yend = row$y.position,
                 linewidth = 0.4, color = "black") +
        annotate("segment",
                 x    = row$x_start, xend = row$x_start,
                 y    = row$y.position,
                 yend = row$y.position - row$y.position * 0.025,
                 linewidth = 0.4, color = "black") +
        annotate("segment",
                 x    = row$x_end, xend = row$x_end,
                 y    = row$y.position,
                 yend = row$y.position - row$y.position * 0.025,
                 linewidth = 0.4, color = "black") +
        annotate("text",
                 x     = (row$x_start + row$x_end) / 2,
                 y     = row$y.position * 1.025,
                 label = row$p.adj.signif,
                 size  = 3.8, color = "black")
    }
  }
  p
}

ct_list      <- c("Micro_2","Astro_2","Astro_3","NVU_unit")
plots_single <- setNames(lapply(ct_list, plot_single_manual), ct_list)

for (ct in ct_list) {
  safe_ct <- gsub("_", "", ct)
  ggsave(paste0("density_single_", safe_ct, ".pdf"),
         plots_single[[ct]], width = 8, height = 5, dpi = 300)
  ggsave(paste0("density_single_", safe_ct, ".png"),
         plots_single[[ct]], width = 8, height = 5, dpi = 300)
}

# ════════════════════════════════════════════════════════════
# 11. 保存结果
# ════════════════════════════════════════════════════════════

write.csv(summary_all,        "density_summary.csv",      row.names = FALSE)
write.csv(all_combined,       "density_combined.csv",     row.names = FALSE)
write.csv(all_ctrl_centers,   "ctrl_centers.csv",         row.names = FALSE)
write.csv(plaque_single_area, "plaque_single_area.csv",   row.names = FALSE)
write.csv(plaque_bin_area,    "plaque_bin_area.csv",      row.names = FALSE)

message("\n=== 完成 ===")

In [ ]:
# ════════════════════════════════════════════════════════════
# 保存画图所需数据（供 Python 重绘使用）
# ════════════════════════════════════════════════════════════

# 1. 柱子均值 + SEM（主绘图数据）
plot_data_bar <- summary_all %>%
  filter(mode == "single",
         subcelltype %in% c("Micro_2","Astro_2","Astro_3","NVU_unit")) %>%
  select(subcelltype, group, area_m, mean_density, sem, n_samples)

write.csv(plot_data_bar,
          "plot_data_bar.csv",
          row.names = FALSE)

# 2. 每个样本的原始均值（用于统计检验 / jitter 点）
plot_data_raw <- per_sample %>%
  filter(mode == "single",
         subcelltype %in% c("Micro_2","Astro_2","Astro_3","NVU_unit")) %>%
  select(subcelltype, group, area_m, sample, sample_mean)

write.csv(plot_data_raw,
          "plot_data_raw.csv",
          row.names = FALSE)

# 3. 统计检验结果（两两比较 + BH 矫正）
pairs <- list(c("AD-Abeta",  "AD-Control"),
              c("AD-Abeta",  "Con-Control"),
              c("AD-Control","Con-Control"))

stat_all <- per_sample %>%
  filter(mode == "single",
         subcelltype %in% c("Micro_2","Astro_2","Astro_3","NVU_unit"),
         !is.na(sample_mean)) %>%
  mutate(area_m      = as.character(area_m),
         group       = as.character(group),
         subcelltype = as.character(subcelltype)) %>%
  group_by(subcelltype, area_m) %>%
  group_modify(function(df, key) {
    purrr::map_dfr(pairs, function(pr) {
      g1 <- df$sample_mean[df$group == pr[1]]
      g2 <- df$sample_mean[df$group == pr[2]]
      if (length(g1) < 2 || length(g2) < 2 ||
          all(is.na(g1)) || all(is.na(g2)) ||
          length(unique(c(g1, g2))) < 2) return(NULL)
      tryCatch(
        tibble(group1 = pr[1], group2 = pr[2],
               p = wilcox.test(g1, g2, exact = FALSE)$p.value),
        error = function(e) NULL)
    })
  }) %>%
  ungroup() %>%
  group_by(subcelltype) %>%           # 按 subcelltype 分组 BH 矫正
  mutate(p.adj = p.adjust(p, method = "BH")) %>%
  ungroup() %>%
  mutate(
    p.adj.signif = case_when(
      p.adj < 0.001 ~ "***",
      p.adj < 0.01  ~ "**",
      p.adj < 0.05  ~ "*",
      TRUE          ~ "ns")
  )

write.csv(stat_all,
          "plot_data_stats.csv",
          row.names = FALSE)

message("已保存:")
message("  plot_data_bar.csv   — ", nrow(plot_data_bar), " 行")
message("  plot_data_raw.csv   — ", nrow(plot_data_raw), " 行")
message("  plot_data_stats.csv — ", nrow(stat_all), " 行")

### Abeta 周围 Endo Perictye的密度变化

In [ ]:
AD_samples <- c("D03556C4", "D01574C4", "C01834C3", "D04303A6",
                "D03556E4", "D01574B6", "D03556D4", "D04303D1",
                "D03556E6", "C01840B1")#,'AD_1','AD_2'

Con_samples <- unique(c("D01574A6", "D01574B1", #"D01574C2", "D01574C6",
                         "B03421A5", "B03421A6", "D03556D6", "D03556E2",
                         "D04305A6", "D04305A4", "C04595E2", "C04595F1",
                         "D03556F6", "D03556F4",'control_1','control_2'))#

AD_cells <- lapply(AD_samples, assign_dist_AD) %>% bind_rows()

Con_cells <- all_cells_raw %>%
  filter(sample %in% Con_samples, area_m %in% target_regions) %>%
  mutate(nearest_plaque   = NA_character_,
         min_dist_to_edge = Inf,
         bin_label        = NA_character_)

In [ ]:
# ════════════════════════════════════════════════════════════
# 针对 Endo/Pericyte 单独设定更大的距离范围
# 500µm，每100µm一个bin，共5个bin
# ════════════════════════════════════════════════════════════

vessel_bin_width_um    <- 50
vessel_max_um          <- 250
vessel_n_bins          <- vessel_max_um / vessel_bin_width_um   # = 5
vessel_bin_width_coord <- vessel_bin_width_um / coord_to_um    # = 200坐标单位
vessel_max_coord       <- vessel_max_um / coord_to_um          # = 1000坐标单位

vessel_bin_breaks <- seq(0, vessel_max_coord, by = vessel_bin_width_coord)
vessel_bin_labels <- paste0(
  seq(0,   vessel_max_um - vessel_bin_width_um, by = vessel_bin_width_um), "-",
  seq(vessel_bin_width_um, vessel_max_um,        by = vessel_bin_width_um), "µm"
)

vessel_bin_areas <- tibble(
  bin_label   = vessel_bin_labels,
  r_out_um    = seq(vessel_bin_width_um, vessel_max_um, by = vessel_bin_width_um),
  r_in_um     = seq(0, vessel_max_um - vessel_bin_width_um, by = vessel_bin_width_um)
) %>%
  mutate(bin_area_um2 = pi * r_out_um^2 - pi * r_in_um^2)

assign_vessel_bin <- function(dist_coord) {
  idx <- findInterval(dist_coord, vessel_bin_breaks,
                      rightmost.closed = FALSE, left.open = TRUE)
  ifelse(dist_coord <= 0 | idx < 1 | idx > vessel_n_bins,
         NA_character_, vessel_bin_labels[idx])
}

vessel_types  <- c("Endo", "Pericyte")
vessel_colors <- c("Endo" = "#2CA25F", "Pericyte" = "#E08214")
# ════════════════════════════════════════════════════════════
# 工具函数：IQR法去除离群值
# 在 per-plaque/per-center 密度层面过滤
# ════════════════════════════════════════════════════════════

remove_outliers_iqr <- function(df, value_col = "density",
                                 group_cols, k = 1.5) {
  df %>%
    group_by(across(all_of(group_cols))) %>%
    mutate(
      q1     = quantile(.data[[value_col]], 0.25, na.rm = TRUE),
      q3     = quantile(.data[[value_col]], 0.75, na.rm = TRUE),
      iqr    = q3 - q1,
      lower  = q1 - k * iqr,
      upper  = q3 + k * iqr,
      is_out = .data[[value_col]] < lower | .data[[value_col]] > upper
    ) %>%
    filter(!is_out) %>%
    select(-q1, -q3, -iqr, -lower, -upper, -is_out) %>%
    ungroup()
}

# ════════════════════════════════════════════════════════════
# 1. AD-Abeta（替换原来的 vessel_abeta）
# ════════════════════════════════════════════════════════════

vessel_abeta <- AD_cells %>%
  filter(
    subcelltype %in% vessel_types,
    area_m %in% c("CA1","SLRM","FAS"),
    !is.na(min_dist_to_edge), !is.infinite(min_dist_to_edge),
    !is.na(nearest_plaque)
  ) %>%
  mutate(bin_label = assign_vessel_bin(min_dist_to_edge)) %>%
  filter(!is.na(bin_label)) %>%
  group_by(sample, area_m, nearest_plaque, bin_label, subcelltype) %>%
  summarise(n = n(), .groups = "drop") %>%
  complete(nesting(sample, area_m, nearest_plaque, bin_label),
           subcelltype = vessel_types, fill = list(n = 0)) %>%
  left_join(vessel_bin_areas %>% select(bin_label, bin_area_um2),
            by = "bin_label") %>%
  filter(!is.na(bin_area_um2), bin_area_um2 > 0) %>%
  mutate(density = n / bin_area_um2 * 1e6) %>%
  # ── 去除离群值（per area_m × bin × subcelltype 组内）────────
  remove_outliers_iqr(
    value_col  = "density",
    group_cols = c("area_m", "bin_label", "subcelltype"),
    k          = 1.25
  ) %>%
  # ── 两步聚合 ─────────────────────────────────────────────────
  group_by(sample, area_m, bin_label, subcelltype) %>%
  summarise(density = mean(density, na.rm = TRUE), .groups = "drop") %>%
  group_by(area_m, bin_label, subcelltype) %>%
  summarise(
    mean_density = mean(density, na.rm = TRUE),
    sem          = sd(density,   na.rm = TRUE) / sqrt(n()),
    n_samples    = n(),
    .groups      = "drop"
  ) %>%
  mutate(group = "AD-Abeta")

# ════════════════════════════════════════════════════════════
# 2. Con-Control（替换原来的 vessel_con）
# ════════════════════════════════════════════════════════════

vessel_con <- lapply(seq_len(nrow(Con_ctrl_centers)), function(i) {
  row <- Con_ctrl_centers[i, ]
  s   <- row$sample; cx <- row$ctrl_x; cy <- row$ctrl_y

  Con_cells %>%
    filter(sample == s,
           area_m %in% c("CA1","SLRM","FAS"),
           subcelltype %in% vessel_types) %>%
    mutate(
      dist      = sqrt((x - cx)^2 + (y - cy)^2),
      bin_label = assign_vessel_bin(dist)
    ) %>%
    filter(!is.na(bin_label)) %>%
    group_by(area_m, bin_label, subcelltype) %>%
    summarise(n = n(), .groups = "drop") %>%
    complete(nesting(area_m, bin_label),
             subcelltype = vessel_types, fill = list(n = 0)) %>%
    left_join(vessel_bin_areas %>% select(bin_label, bin_area_um2),
              by = "bin_label") %>%
    filter(!is.na(bin_area_um2), bin_area_um2 > 0) %>%
    mutate(density   = n / bin_area_um2 * 1e6,
           sample    = s,
           center_id = row$center_id)
}) %>%
  bind_rows() %>%
  # ── 去除离群值 ────────────────────────────────────────────────
  remove_outliers_iqr(
    value_col  = "density",
    group_cols = c("area_m", "bin_label", "subcelltype"),
    k          = 1.25
  ) %>%
  # ── 两步聚合 ─────────────────────────────────────────────────
  group_by(sample, area_m, bin_label, subcelltype) %>%
  summarise(density = mean(density, na.rm = TRUE), .groups = "drop") %>%
  group_by(area_m, bin_label, subcelltype) %>%
  summarise(
    mean_density = mean(density, na.rm = TRUE),
    sem          = sd(density,   na.rm = TRUE) / sqrt(n()),
    n_samples    = n(),
    .groups      = "drop"
  ) %>%
  mutate(group = "Con-Control")

# ── 诊断：去除了多少 ──────────────────────────────────────────
message("vessel_abeta n_samples 分布（去除后）:")
print(vessel_abeta %>% select(area_m, bin_label, subcelltype, n_samples))
message("vessel_con n_samples 分布（去除后）:")
print(vessel_con %>% select(area_m, bin_label, subcelltype, n_samples))

# ════════════════════════════════════════════════════════════
# 3. 合并（与原代码相同）
# ════════════════════════════════════════════════════════════

vessel_all <- bind_rows(vessel_abeta, vessel_con) %>%
  mutate(
    group       = factor(group, levels = c("Con-Control","AD-Abeta")),
    area_m      = factor(area_m, levels = c("CA1","SLRM","FAS")),
    subcelltype = factor(subcelltype, levels = vessel_types),
    bin_start   = as.numeric(sub("(\\d+)-\\d+µm","\\1",
                                  as.character(bin_label))),
    bin_mid     = bin_start + vessel_bin_width_um / 2
  )

# 画图代码不变，直接跑 p_vessel 和 ggsave

In [ ]:
# ════════════════════════════════════════════════════════════
# 重写：直接计算 count，不复用旧函数
# ════════════════════════════════════════════════════════════

# ── AD-Abeta：重新用 assign_vessel_bin 分bin ─────────────────
count_abeta <- AD_cells %>%
  filter(
    subcelltype %in% vessel_types,
    area_m %in% c("CA1","SLRM","FAS"),
    !is.na(min_dist_to_edge), !is.infinite(min_dist_to_edge),
    !is.na(nearest_plaque)
  ) %>%
  # 关键：用 vessel bin，不用旧的 bin_label 列
  mutate(vbin = assign_vessel_bin(min_dist_to_edge)) %>%
  filter(!is.na(vbin)) %>%
  group_by(sample, area_m, nearest_plaque, vbin, subcelltype) %>%
  summarise(n = n(), .groups = "drop") %>%
  complete(nesting(sample, area_m, nearest_plaque, vbin),
           subcelltype = vessel_types, fill = list(n = 0)) %>%
  # per-plaque → per-sample
  group_by(sample, area_m, vbin, subcelltype) %>%
  summarise(n = mean(n, na.rm = TRUE), .groups = "drop") %>%
  # per-sample → 跨sample
  group_by(area_m, vbin, subcelltype) %>%
  summarise(mean_n = mean(n, na.rm = TRUE),
            sem    = sd(n,   na.rm = TRUE) / sqrt(n()),
            .groups = "drop") %>%
  mutate(group = "AD-Abeta")

message("count_abeta:")
print(count_abeta)

# ── Con-Control ───────────────────────────────────────────────
count_con <- lapply(seq_len(nrow(Con_ctrl_centers)), function(i) {
  row <- Con_ctrl_centers[i, ]
  s   <- row$sample; cx <- row$ctrl_x; cy <- row$ctrl_y

  Con_cells %>%
    filter(sample == s,
           area_m %in% c("CA1","SLRM","FAS"),
           subcelltype %in% vessel_types) %>%
    mutate(
      dist = sqrt((x - cx)^2 + (y - cy)^2),
      vbin = assign_vessel_bin(dist)
    ) %>%
    filter(!is.na(vbin)) %>%
    group_by(area_m, vbin, subcelltype) %>%
    summarise(n = n(), .groups = "drop") %>%
    complete(nesting(area_m, vbin),
             subcelltype = vessel_types, fill = list(n = 0)) %>%
    mutate(sample = s, center_id = row$center_id)
}) %>%
  bind_rows() %>%
  group_by(sample, area_m, vbin, subcelltype) %>%
  summarise(n = mean(n, na.rm = TRUE), .groups = "drop") %>%
  group_by(area_m, vbin, subcelltype) %>%
  summarise(mean_n = mean(n, na.rm = TRUE),
            sem    = sd(n,   na.rm = TRUE) / sqrt(n()),
            .groups = "drop") %>%
  mutate(group = "Con-Control")

message("count_con:")
print(count_con)

# ── Fold-change ───────────────────────────────────────────────
fc_vessel <- count_abeta %>%
  rename(mean_n_ad = mean_n, sem_ad = sem) %>%
  left_join(
    count_con %>% rename(mean_n_con = mean_n, sem_con = sem),
    by = c("area_m","vbin","subcelltype")
  ) %>%
  filter(!is.na(mean_n_con), mean_n_con > 0) %>%
  mutate(
    fc      = mean_n_ad / mean_n_con,
    fc_sem  = fc * sqrt((sem_ad  / pmax(mean_n_ad,  0.001))^2 +
                          (sem_con / pmax(mean_n_con, 0.001))^2),
    bin_start   = as.numeric(sub("(\\d+)-\\d+µm","\\1", as.character(vbin))),
    bin_mid     = bin_start + vessel_bin_width_um / 2,
    area_m      = factor(area_m,      levels = c("CA1","SLRM","FAS")),
    subcelltype = factor(subcelltype, levels = vessel_types)
  )

message("\nfc_vessel:")
print(fc_vessel %>% select(area_m, vbin, subcelltype, mean_n_ad,
                             mean_n_con, fc) %>% arrange(area_m, vbin))

# ── 画图 ──────────────────────────────────────────────────────
p_fc <- ggplot(fc_vessel,
               aes(x = bin_mid, y = fc,
                   color = subcelltype, fill = subcelltype)) +
  geom_hline(yintercept = 1, linetype = "dashed",
             color = "grey50", linewidth = 0.4) +
  geom_ribbon(aes(ymin = pmax(fc - fc_sem, 0), ymax = fc + fc_sem),
              alpha = 0.15, color = NA) +
  geom_line(linewidth = 0.85) +
  geom_point(size = 2.2, shape = 21, stroke = 0.4, color = "white") +
  facet_wrap(~ area_m, nrow = 1) +
  scale_color_manual(values = vessel_colors) +
  scale_fill_manual(values  = vessel_colors) +
  scale_x_continuous(
    breaks = x_breaks, labels = x_labels,
    expand = expansion(mult = c(0.02, 0.02))
  ) +
  scale_y_continuous(expand = expansion(mult = c(0.05, 0.1))) +
  labs(
    # ── 关键：用 expression() 处理希腊字母 ──────────────────
    title    = expression("Vessel cell enrichment around A"*beta*" plaques"),
    subtitle = "Fold-change vs Con-Control; dashed line = no change (FC=1)",
    x        = expression("Distance from plaque edge ("*mu*"m)"),
    y        = "Fold-change (AD-Abeta / Con-Control)",
    color    = NULL, fill = NULL
  ) +
  theme_classic(base_size = 11) +
  theme(
    axis.text.x        = element_text(angle = 45, hjust = 1,
                                       size = 9, color = "black"),
    axis.text.y        = element_text(size = 9, color = "black"),
    axis.line          = element_line(linewidth = 0.4),
    axis.ticks         = element_line(linewidth = 0.35),
    strip.background   = element_blank(),
    strip.text         = element_text(face = "bold", size = 10),
    legend.position    = "bottom",
    panel.grid.major.y = element_line(linewidth = 0.2, color = "grey88"),
    panel.spacing      = unit(0.5, "cm"),
    plot.title         = element_text(face = "bold", size = 12, hjust = 0),
    plot.subtitle      = element_text(size = 9, color = "grey40", hjust = 0),
    plot.margin        = margin(10, 10, 6, 10)
  )

# ── PDF用cairo_pdf避免字符编码问题 ──────────────────────────
cairo_pdf("vessel_foldchange.pdf", width = 10, height = 5)
print(p_fc)
dev.off()

ggsave("vessel_foldchange.png", p_fc,
       width = 10, height = 5, dpi = 300)

message("完成: vessel_foldchange.pdf/.png")

### Aβ 累计程度 ✅️

In [ ]:
head(all_abeta_filled)

In [ ]:
plaque_burden <- all_abeta_filled %>%
  group_by(sample, plaque_id) %>%
  summarise(plaque_area_um2 = n() * coord_to_um2, .groups = "drop") %>%
  # 需要知道每个斑块落在哪个 area_m
  # 方法：用斑块质心做KNN匹配到最近的细胞，取该细胞的area_m
  left_join(all_centroids, by = c("sample", "plaque_id")) %>%
  {
    cells_ref <- all_cells_raw %>%
      filter(sample %in% AD_samples) %>%
      select(sample, x, y, area_m)
    # 对每个斑块质心找最近细胞的area_m
    nn <- FNN::get.knnx(
      data  = as.matrix(cells_ref[, c("x","y")]),
      query = as.matrix(.[, c("cx","cy")]), k = 1
    )
    mutate(., area_m = cells_ref$area_m[nn$nn.index[,1]])
  } %>%
  group_by(sample, area_m) %>%
  summarise(
    n_plaques       = n(),
    total_area_um2  = sum(plaque_area_um2),
    mean_area_um2   = mean(plaque_area_um2),
    .groups = "drop"
  )

# 可视化
ggplot(plaque_burden, aes(x = area_m, y = n_plaques)) +
  geom_boxplot(fill = "#E64B35", alpha = 0.7, outlier.shape = 21) +
  geom_jitter(width = 0.15, alpha = 0.5, size = 1.5) +
  labs(x = "Brain region", y = "Number of plaques per sample",
       title = "Aβ plaque burden by hippocampal subregion") +
  theme_classic(base_size = 13)

In [ ]:
library(ggbeeswarm)
library(ggsignif)

area_palette <- c(
  "CA1"  = "#E64B35",
  "CA2"  = "#F39B7F",
  "CA3"  = "#FFCD00",
  "CA4"  = "#00A087",
  "DG"   = "#3C5488",
  "FAS"  = "#8491B4",
  "SLRM" = "#91D1C2"
)

theme_pub <- function() {
  theme_classic(base_size = 11, base_family = "sans") +
    theme(
      axis.line         = element_line(linewidth = 0.4, color = "black"),
      axis.ticks        = element_line(linewidth = 0.4, color = "black"),
      axis.ticks.length = unit(2.5, "pt"),
      axis.text         = element_text(color = "black", size = 10),
      axis.title        = element_text(color = "black", size = 11),
      plot.title        = element_text(size = 11, face = "bold", hjust = 0),
      plot.tag          = element_text(size = 12, face = "bold"),
      legend.position   = "none",
      panel.grid        = element_blank(),
      plot.background   = element_blank(),
      panel.background  = element_blank()
    )
}

run_wilcox_pairs <- function(df, var, pairs) {
  purrr::map_dfr(pairs, function(pr) {
    vals1 <- df[[var]][df$area_m == pr[1]]
    vals2 <- df[[var]][df$area_m == pr[2]]
    if (length(vals1) < 2 || length(vals2) < 2) return(NULL)
    p <- wilcox.test(vals1, vals2, exact = FALSE)$p.value
    tibble(
      group1   = pr[1],
      group2   = pr[2],
      p.value  = p,
      p.signif = case_when(
        p < 0.001 ~ "***",
        p < 0.01  ~ "**",
        p < 0.05  ~ "*",
        TRUE      ~ "ns"
      )
    )
  }) %>%
    filter(p.signif != "ns")
}

focal_pairs <- list(
  c("CA1",  "CA2"),  c("CA1",  "CA3"),  c("CA1",  "CA4"),
  c("CA1",  "SLRM"), c("CA1",  "FAS"),
  c("SLRM", "CA2"),  c("SLRM", "CA3"),  c("SLRM", "CA4"),
  c("FAS",  "CA2"),  c("FAS",  "CA3"),  c("FAS",  "CA4"),
  c("SLRM", "FAS")
)

# ── 关键修改1：过滤 NA，并将 area_m 转为有序 factor ──────────
area_levels <- c("CA1", "CA2", "CA3", "CA4", "DG", "SLRM", "FAS")

plaque_burden <- plaque_burden %>%
  filter(!is.na(area_m), area_m != "NA", area_m %in% names(area_palette)) %>%
  mutate(area_m = factor(area_m, levels = area_levels))

sig_n     <- run_wilcox_pairs(plaque_burden, "n_plaques",      focal_pairs)
sig_total <- run_wilcox_pairs(plaque_burden, "total_area_um2", focal_pairs)
sig_mean  <- run_wilcox_pairs(plaque_burden, "mean_area_um2",  focal_pairs)

make_plot <- function(df, yvar, ylab, title, tag, sig_df,
                      y_expand = 0.25) {
  y_max   <- max(df[[yvar]], na.rm = TRUE)
  y_range <- diff(range(df[[yvar]], na.rm = TRUE))
  n_sig   <- if (!is.null(sig_df) && nrow(sig_df) > 0) nrow(sig_df) else 0

  # ── 关键修改2：geom_signif manual 模式正确用法 ──────────────
  if (n_sig > 0) {
    sig_df <- sig_df %>%
      mutate(
        x1   = match(as.character(group1), levels(df$area_m)),
        x2   = match(as.character(group2), levels(df$area_m)),
        span = abs(x2 - x1)
      ) %>%
      arrange(span) %>%
      mutate(
        y_position = y_max + y_range * (0.08 + 0.10 * (row_number() - 1))
      )
    
    # 构建 comparisons list 和对应 annotations/y_position 向量
    comp_list  <- purrr::map2(sig_df$group1, sig_df$group2, c)
    annot_vec  <- sig_df$p.signif
    ypos_vec   <- sig_df$y_position
  }

  p <- ggplot(df, aes(x = area_m, y = .data[[yvar]],
                      color = area_m, fill = area_m)) +
    geom_boxplot(
      width = 0.5, linewidth = 0.45,
      outlier.shape = NA,
      alpha = 0.15, color = "black", fill = "white"
    ) +
    stat_summary(
      fun.data = function(x)
        tibble(ymin = quantile(x, 0.25), y = median(x),
               ymax = quantile(x, 0.75)),
      geom = "crossbar", width = 0.5, linewidth = 0.45,
      aes(fill = area_m), alpha = 0.5, color = NA
    ) +
    stat_summary(
      fun = median, geom = "crossbar",
      width = 0.5, linewidth = 0.7,
      color = "black", fatten = 1
    ) +
    geom_beeswarm(
      size = 2.5, cex = 2.5, alpha = 0.85,
      shape = 21, stroke = 0.4,
      aes(fill = area_m), color = "white"
    ) +
    stat_summary(
      fun = mean, geom = "point",
      shape = 23, size = 2.5, stroke = 0.6,
      fill = "white", color = "black"
    ) +
    scale_color_manual(values = area_palette, drop = TRUE) +
    scale_fill_manual(values  = area_palette, drop = TRUE) +
    scale_y_continuous(
      expand = expansion(mult = c(0.03, y_expand)),
      labels = scales::label_comma()
    ) +
    labs(x = NULL, y = ylab, title = title, tag = tag) +
    theme_pub()

  # ── 关键修改3：用 comparisons 模式替代 manual ────────────────
  if (n_sig > 0) {
    p <- p + ggsignif::geom_signif(
      comparisons  = comp_list,
      annotations  = annot_vec,
      y_position   = ypos_vec,
      tip_length   = 0.008,
      textsize     = 2.8,
      vjust        = 0.3,
      color        = "black",
      size         = 0.35        # 旧版 ggsignif 用 size，非 linewidth
    )
  }
  p
}

# ── 单独保存 p2 ──────────────────────────────────────────────
p2 <- make_plot(plaque_burden, "total_area_um2",
                "Total plaque area (\u00b5m\u00b2)",   # 修改4：避免 beta 符号乱码
                "Total plaque burden", "B", sig_total)

ggsave("p2_total_plaque_burden.pdf", p2,
       width = 8.5, height = 6, useDingbats = FALSE, dpi = 300)
ggsave("p2_total_plaque_burden.png", p2,
       width =8.5, height = 6, dpi = 300)

# ── 生成三图组合（标题也改用 unicode 避免乱码）───────────────
p1 <- make_plot(plaque_burden, "n_plaques",
                "Number of plaques", "Plaque count", "A", sig_n)
p3 <- make_plot(plaque_burden, "mean_area_um2",
                "Mean plaque area (\u00b5m\u00b2)", "Mean plaque size", "C", sig_mean)

fig <- p1 / p2 +
  plot_annotation(
    title   = "A\u03b2 plaque burden across hippocampal subregions",  # Aβ unicode
    caption = "Wilcoxon rank-sum test; *p<0.05, **p<0.01, ***p<0.001\nDiamond = mean; line = median",
    theme   = theme(
      plot.title   = element_text(size = 12, face = "bold", hjust = 0),
      plot.caption = element_text(size = 8, color = "grey50", hjust = 0)
    )
  )

ggsave("plaque_burden_pub.pdf", fig,
       width = 7, height = 15, useDingbats = FALSE, dpi = 300)
ggsave("plaque_burden_pub.png", fig,
       width = 8, height = 15, dpi = 300)

### Abeta 周围 Micro Astro 比例变化

In [ ]:
# ════════════════════════════════════════════════════════════
# 参数更新：40µm bin，0-200µm
# ════════════════════════════════════════════════════════════

bin_width_um    <- 50
max_um          <- 250
n_bins          <- max_um / bin_width_um          # = 5
bin_width_coord <- bin_width_um / coord_to_um     # = 80坐标单位
max_coord       <- max_um / coord_to_um           # = 400坐标单位

bin_breaks_coord <- seq(0, max_coord, by = bin_width_coord)
bin_labels <- paste0(
  seq(0,   max_um - bin_width_um, by = bin_width_um), "-",
  seq(bin_width_um, max_um,       by = bin_width_um), "µm"
)
# "0-40µm" "40-80µm" "80-120µm" "120-160µm" "160-200µm"

# assign_bin 重新定义（参数已更新，直接用全局变量）
assign_bin <- function(dist_coord) {
  idx <- findInterval(dist_coord, bin_breaks_coord,
                      rightmost.closed = FALSE, left.open = TRUE)
  ifelse(dist_coord <= 0 | idx < 1 | idx > n_bins,
         NA_character_, bin_labels[idx])
}

# ── 重新跑 AD_cells 的 bin_label（距离不变，只重新分bin）────
AD_cells <- AD_cells %>%
  mutate(bin_label = assign_bin(min_dist_to_edge))

# ════════════════════════════════════════════════════════════
# 颜色
# ════════════════════════════════════════════════════════════

micro_colors <- c(
  "Micro_0" = "#DADAEB", "Micro_1" = "#9E9AC8",
  "Micro_2" = "#3F007D", "Micro_3" = "#6A51A3",
  "Micro_4" = "#BCBDDC"
)
astro_colors <- c(
  "Astro_0" = "#4292C6", "Astro_1" = "#74C0D8",
  "Astro_2" = "#08306B", "Astro_3" = "#2171B5",
  "Astro_4" = "#C6DBEF"
)

micro_subtypes <- c("Micro_0","Micro_1","Micro_2","Micro_3","Micro_4")
astro_subtypes <- c("Astro_0","Astro_1","Astro_2","Astro_3","Astro_4")

# ════════════════════════════════════════════════════════════
# 数据计算：三组
# AD-Abeta    : 真实斑块，形态学距离
# Con-Control : Con样本随机中心点
# 组别顺序：Con-Control（左）| AD-Abeta（右）
# ════════════════════════════════════════════════════════════

# ── AD-Abeta ────────────────────────────────────────────────
calc_abeta_prop <- function(subtypes) {
  AD_cells %>%
    filter(
      subcelltype %in% subtypes,
      area_m %in% c("CA1","SLRM","FAS"),
      !is.na(bin_label), !is.na(nearest_plaque)
    ) %>%
    group_by(sample, area_m, nearest_plaque, bin_label, subcelltype) %>%
    summarise(n = n(), .groups = "drop") %>%
    complete(nesting(sample, area_m, nearest_plaque, bin_label),
             subcelltype = subtypes, fill = list(n = 0)) %>%
    group_by(sample, area_m, nearest_plaque, bin_label) %>%
    mutate(total = sum(n),
           prop  = ifelse(total > 0, n / total, NA_real_)) %>%
    ungroup() %>%
    group_by(sample, area_m, bin_label, subcelltype) %>%
    summarise(prop = mean(prop, na.rm = TRUE), .groups = "drop") %>%
    group_by(area_m, bin_label, subcelltype) %>%
    summarise(mean_prop = mean(prop, na.rm = TRUE), .groups = "drop") %>%
    mutate(group = "AD-Abeta")
}

# ── Con-Control ──────────────────────────────────────────────
calc_con_prop <- function(subtypes) {
  result <- vector("list", nrow(Con_ctrl_centers))
  for (i in seq_len(nrow(Con_ctrl_centers))) {
    row <- Con_ctrl_centers[i, ]
    s   <- row$sample; cx <- row$ctrl_x; cy <- row$ctrl_y

    cells_s <- Con_cells %>%
      filter(sample == s,
             area_m %in% c("CA1","SLRM","FAS"),
             subcelltype %in% subtypes) %>%
      mutate(
        dist      = sqrt((x - cx)^2 + (y - cy)^2),
        bin_label = assign_bin(dist)
      ) %>%
      filter(!is.na(bin_label))

    if (nrow(cells_s) == 0) next

    result[[i]] <- cells_s %>%
      group_by(area_m, bin_label, subcelltype) %>%
      summarise(n = n(), .groups = "drop") %>%
      complete(nesting(area_m, bin_label),
               subcelltype = subtypes, fill = list(n = 0)) %>%
      group_by(area_m, bin_label) %>%
      mutate(total = sum(n),
             prop  = ifelse(total > 0, n / total, NA_real_),
             sample = s) %>%
      ungroup()
  }
  bind_rows(result) %>%
    group_by(sample, area_m, bin_label, subcelltype) %>%
    summarise(prop = mean(prop, na.rm = TRUE), .groups = "drop") %>%
    group_by(area_m, bin_label, subcelltype) %>%
    summarise(mean_prop = mean(prop, na.rm = TRUE), .groups = "drop") %>%
    mutate(group = "Con-Control")
}

# ── 合并，归一化 ─────────────────────────────────────────────
prepare_prop <- function(subtypes, colors) {
  bind_rows(
    calc_abeta_prop(subtypes),
    calc_con_prop(subtypes)
  ) %>%
    mutate(
      # Con-Control 在左，AD-Abeta 在右
      group       = factor(group, levels = c("Con-Control","AD-Abeta")),
      area_m      = factor(area_m, levels = c("CA1","SLRM","FAS")),
      subcelltype = factor(subcelltype, levels = names(colors)),
      bin_start   = as.numeric(sub("(\\d+)-\\d+µm", "\\1",
                                    as.character(bin_label))),
      bin_mid     = bin_start + bin_width_um / 2
    ) %>%
    group_by(group, area_m, bin_label) %>%
    mutate(mean_prop = mean_prop / sum(mean_prop, na.rm = TRUE)) %>%
    ungroup()
}

prop_micro <- prepare_prop(micro_subtypes, micro_colors)
prop_astro <- prepare_prop(astro_subtypes, astro_colors)

# ════════════════════════════════════════════════════════════
# 画图
# ════════════════════════════════════════════════════════════

theme_pub_stack <- function() {
  theme_classic(base_size = 11) +
  theme(
    axis.text.x        = element_text(angle = 45, hjust = 1,
                                       size = 9, color = "black"),
    axis.text.y        = element_text(size = 9, color = "black"),
    axis.title         = element_text(size = 10),
    axis.line          = element_line(linewidth = 0.4),
    axis.ticks         = element_line(linewidth = 0.35),
    axis.ticks.length  = unit(2.5, "pt"),
    strip.background   = element_blank(),
    strip.text.x       = element_text(face = "bold", size = 10),
    strip.text.y       = element_text(face = "bold", size = 10,
                                       angle = 0, hjust = 0),
    legend.position    = "bottom",
    legend.title       = element_blank(),
    legend.text        = element_text(size = 9),
    legend.key.size    = unit(0.45, "cm"),
    legend.spacing.x   = unit(0.15, "cm"),
    panel.grid.major.y = element_line(linewidth = 0.2, color = "grey88"),
    panel.spacing      = unit(0.5, "cm"),
    plot.title         = element_text(face = "bold", size = 12, hjust = 0),
    plot.tag           = element_text(face = "bold", size = 14),
    plot.margin        = margin(10, 10, 6, 10)
  )
}

make_stack <- function(dat, colors, title_str, tag_str) {
  # x轴刻度：每个bin左边界
  x_breaks <- seq(bin_width_um / 2,
                  max_um - bin_width_um / 2,
                  by = bin_width_um)
  x_labels <- paste0(seq(0, max_um - bin_width_um, by = bin_width_um), "µm")

  ggplot(dat, aes(x = bin_mid, y = mean_prop, fill = subcelltype)) +
    geom_area(position = "fill", alpha = 0.9,
              color = "white", linewidth = 0.25) +
    facet_grid(area_m ~ group) +
    scale_fill_manual(
      values = colors,
      guide  = guide_legend(nrow = 1,
                             keywidth  = unit(0.6, "cm"),
                             keyheight = unit(0.4, "cm"))
    ) +
    scale_x_continuous(
      breaks = x_breaks,
      labels = x_labels,
      expand = expansion(mult = c(0.01, 0.01))
    ) +
    scale_y_continuous(
      labels = scales::percent_format(accuracy = 1),
      expand = expansion(mult = c(0, 0.02)),
      breaks = c(0, 0.25, 0.5, 0.75, 1)
    ) +
    labs(
      title = title_str,
      tag   = tag_str,
      x     = "Distance from plaque edge (µm)",
      y     = "Subtype proportion"
    ) +
    theme_pub_stack()
}

p_micro <- make_stack(prop_micro, micro_colors,
                       "Microglia subtype composition around Aβ plaques", "A")
p_astro <- make_stack(prop_astro, astro_colors,
                       "Astrocyte subtype composition around Aβ plaques", "B")

# ════════════════════════════════════════════════════════════
# 保存
# ════════════════════════════════════════════════════════════

ggsave("stack_micro_40um.pdf", p_micro,
       width = 8, height = 7, dpi = 300, useDingbats = FALSE)
ggsave("stack_micro_40um.png", p_micro,
       width = 8, height = 7, dpi = 300)

ggsave("stack_astro_40um.pdf", p_astro,
       width = 8, height = 7, dpi = 300, useDingbats = FALSE)
ggsave("stack_astro_40um.png", p_astro,
       width = 8, height = 7, dpi = 300)

message("完成: stack_micro_40um / stack_astro_40um")